# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR⁲ dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available Record Sets, fields, and their `@id`s.

> All entity references (record sets, fields, columns) use their `@id`.

In [ ]:
print("# Record Sets (by @id):\n")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- recordSet @id: {rs.id}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.id} (name: {getattr(f, 'name', 'N/A')}, dataType: {getattr(f, 'data_type', 'N/A')})")
    if hasattr(rs, 'columns'):
        print("  Columns:")
        for c in rs.columns:
            print(f"    - {c.id} (name: {getattr(c, 'name', 'N/A')}, dataType: {getattr(c, 'data_type', 'N/A')})")
    print()
# Print the first record of each record set for inspection
for rs in record_sets:
    print(f"First record for recordSet @id: {rs.id}")
    try:
        records_iter = dataset.records(record_set=rs.id)
        first_record = next(records_iter)
        print(first_record)
    except Exception as e:
        print("  No records found or failed to load.")
    print()

## 3. Data Extraction
Load all data from each record set into DataFrames for easy analysis. Record set and field IDs use their Croissant `@id`s.

In [ ]:
# Collect all recordSet @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for Record Set: {record_set_id} - Shape: {df.shape}")
    if not df.empty:
        print(f"   Columns: {df.columns.tolist()[:8]}{' ...' if len(df.columns) > 8 else ''}")
    print()

# Preview the first record set dataframe
if len(record_set_ids) > 0:
    rs_id = record_set_ids[0]
    print(f"Sample data from record set: {rs_id}")
    display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group by key fields.

- We select a numeric field (`amount`, `normalized_abundance`, or similar) by its `@id` (find in printout above, e.g., `'amount'` or `'normalized_abundance'`).
- Adjust IDs as appropriate for this dataset using field IDs from the overview section.

In [ ]:
# --- Set the record set and field `@id` for analysis ---
record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
df = dataframes[record_set_id]

# Try to autodetect a numeric field
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print("No numeric field found for EDA.")
else:
    print(f"Using numeric field: {numeric_field_id}")

    # Drop NAs just in case
    df_nonan = df.dropna(subset=[numeric_field_id])

    # Filter
    threshold = df_nonan[numeric_field_id].mean()
    filtered_df = df_nonan[df_nonan[numeric_field_id] > threshold]
    print(f"Records with {numeric_field_id} > {threshold:.2f} (mean): {filtered_df.shape[0]}")
    display(filtered_df.head())

    # Normalize
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Sample normalized values:")
    display(filtered_df[[numeric_field_id, col_norm]].head())

    # Attempt group by a categorical field
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field = col
            break
    if group_field is not None:
        print(f"Grouping results by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print("Sample grouped summary:")
        display(grouped_df.head())
    else:
        print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize data distributions and relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30, alpha=0.7)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,5))
        grouped_df.plot.bar(x=group_field, y=numeric_field_id, legend=False)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.title(f'Mean {numeric_field_id} by {group_field}')
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR⁲ dataset for human extracellular vesicle protein analysis using mass spectrometry.

- We listed available record sets, fields, and their `@id`s using the Croissant schema via `mlcroissant`.
- We loaded data, performed basic filtering and normalization on selected numeric fields, examined group-wise summaries, and visualized data distributions.
- This template can be extended for deeper biological and bioinformatics investigations of the protein dataset.

Remember to always reference fields and record sets by their Croissant `@id` when manipulating data.